In [3]:
!pip install xgboost lightgbm pyarrow joblib shap -q

In [2]:
import pandas as pd
import numpy as np
import time
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

from xgboost import XGBClassifier

In [4]:
import os
import glob

dataset_path = "archive.zip"

files = glob.glob(os.path.join(dataset_path, "*.parquet"))

print(files)

[]


In [5]:
import zipfile

zip_path = "archive.zip"

extract_path = "CICIDS2017"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction Complete!")

BadZipFile: File is not a zip file

In [6]:
import os

print(os.path.exists("archive.zip"))
print(os.path.getsize("archive.zip"))

True
237925705


In [7]:
!file archive.zip

zsh:1: command not found: file


In [8]:
!ls -lh

total 227M
-rw-r--r-- 1 srangoonwala srangoonwala 6.6K Jul 24 15:01 Untitled.ipynb
-rw-r--r-- 1 srangoonwala srangoonwala 227M Jul 24 15:00 archive.zip
-rwxr--r-- 1 srangoonwala srangoonwala   33 Jul 24 14:55 main.py


In [9]:
with open("archive.zip", "rb") as f:
    print(f.read(16))

b'PK\x03\x04-\x00\x00\x00\x08\x00j\xb0\x0bUd\xa3'


In [10]:
import zipfile

print(zipfile.is_zipfile("archive.zip"))

True


In [11]:
import zipfile

with zipfile.ZipFile("archive.zip", "r") as zip_ref:
    print("Number of files:", len(zip_ref.namelist()))
    print("\nFirst 20 files:")
    for file in zip_ref.namelist()[:20]:
        print(file)

Number of files: 8

First 20 files:
Benign-Monday-no-metadata.parquet
Botnet-Friday-no-metadata.parquet
Bruteforce-Tuesday-no-metadata.parquet
DDoS-Friday-no-metadata.parquet
DoS-Wednesday-no-metadata.parquet
Infiltration-Thursday-no-metadata.parquet
Portscan-Friday-no-metadata.parquet
WebAttacks-Thursday-no-metadata.parquet


In [12]:
import zipfile

extract_path = "./CICIDS2017"

with zipfile.ZipFile("archive.zip", "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction Complete!")

Extraction Complete!


In [13]:
import os

for root, dirs, files in os.walk("CICIDS2017"):
    print(root)
    for f in files[:10]:
        print("   ", f)

CICIDS2017
    WebAttacks-Thursday-no-metadata.parquet
    Benign-Monday-no-metadata.parquet
    Infiltration-Thursday-no-metadata.parquet
    Bruteforce-Tuesday-no-metadata.parquet
    DoS-Wednesday-no-metadata.parquet
    Botnet-Friday-no-metadata.parquet
    Portscan-Friday-no-metadata.parquet
    DDoS-Friday-no-metadata.parquet


In [14]:
import glob
import os
import pandas as pd

dataset_path = "CICIDS2017"

files = glob.glob(os.path.join(dataset_path, "*.parquet"))

print(f"Found {len(files)} parquet files")

df = pd.concat(
    [pd.read_parquet(file) for file in files],
    ignore_index=True
)

print(df.shape)

Found 8 parquet files
(2313810, 78)


In [15]:
print("Original Shape:", df.shape)

df.drop_duplicates(inplace=True)

print("After Removing Duplicates:", df.shape)

Original Shape: (2313810, 78)
After Removing Duplicates: (2231806, 78)


In [16]:
import numpy as np

df.replace([np.inf, -np.inf], np.nan, inplace=True)

,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,6,113095465,48,24,9668,10012,403,0,201.416672,203.548294,...,32,203985.500,5.758372e+05,1629110,379,13800000.0,4277541.0,16500000,6737603,Benign
1,6,113473706,68,40,11364,12718,403,0,167.117645,171.919418,...,32,178326.875,5.034269e+05,1424245,325,13800000.0,4229413.0,16500000,6945512,Benign
2,0,119945515,150,0,0,0,0,0,0.000000,0.000000,...,0,6909777.500,1.170000e+07,20400000,6,24400000.0,24300000.0,60100000,5702188,Benign
3,6,60261928,9,7,2330,4221,1093,0,258.888885,409.702148,...,20,0.000,0.000000e+00,0,0,0.0,0.0,0,0,Benign
4,17,269,2,2,102,322,51,51,51.000000,0.000000,...,32,0.000,0.000000e+00,0,0,0.0,0.0,0,0,Benign
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2313799,6,4,2,0,0,0,0,0,0.000000,0.000000,...,32,0.000,0.000000e+00,0,0,0.0,0.0,0,0,Benign
2313801,6,22,1,1,0,0,0,0,0.000000,0.000000,...,32,0.000,0.000000e+00,0,0,0.0,0.0,0,0,Benign
2313803,6,45,1,1,0,0,0,0,0.000000,0.000000,...,32,0.000,0.000000e+00,0,0,0.0,0.0,0,0,Benign
2313808,6,48,2,0,12,0,6,6,6.000000,0.000000,...,20,0.000,0.000000e+00,0,0,0.0,0.0,0,0,Benign


In [17]:
numeric_cols = df.select_dtypes(include=np.number).columns

df[numeric_cols] = df[numeric_cols].fillna(
    df[numeric_cols].median()
)

print(df.isnull().sum().sum())

0


In [18]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

df["Label"] = label_encoder.fit_transform(df["Label"])

print(label_encoder.classes_)

['Benign' 'Bot' 'DDoS' 'DoS GoldenEye' 'DoS Hulk' 'DoS Slowhttptest'
 'DoS slowloris' 'FTP-Patator' 'Heartbleed' 'Infiltration' 'PortScan'
 'SSH-Patator' 'Web Attack � Brute Force' 'Web Attack � Sql Injection'
 'Web Attack � XSS']


In [19]:
X = df.drop("Label", axis=1)
y = df["Label"]

In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(1785444, 77)
(446362, 77)


In [28]:
import xgboost as xgb

print(xgb.__version__)

3.3.0


In [29]:
from xgboost import XGBClassifier

model = XGBClassifier(
    device="cuda",
    tree_method="hist"
)

print(model)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device='cuda', early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)


In [30]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [31]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

X, y = load_iris(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=42
)

model = XGBClassifier(
    device="cuda",
    tree_method="hist",
    n_estimators=10
)

model.fit(X_train, y_train)

print("Training Successful")
print(model.score(X_test, y_test))

Training Successful
1.0


In [63]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import numpy as np
import pandas as pd
import time

results = []

def evaluate_model(model, model_name):

    print("=" * 70)
    print(f"Training {model_name}...")
    print("=" * 70)

    start = time.time()

    # Train
    model.fit(X_train, y_train)

    training_time = time.time() - start

    print("Training Completed!")

    # Predict
    print("Making Predictions...")
    y_pred = model.predict(X_test)

    print("Prediction Completed!")

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )
    recall = recall_score(
        y_test,
        y_pred,
        average="weighted"
    )
    f1 = f1_score(
        y_test,
        y_pred,
        average="weighted"
    )

    print("\n" + "=" * 70)
    print(model_name)
    print("=" * 70)

    print(f"Training Time : {training_time:.2f} sec")
    print(f"Accuracy      : {accuracy:.6f}")
    print(f"Precision     : {precision:.6f}")
    print(f"Recall        : {recall:.6f}")
    print(f"F1 Score      : {f1:.6f}")

    # Display only labels that actually appear
    labels = np.unique(np.concatenate([y_test, y_pred]))
    target_names = label_encoder.inverse_transform(labels)

    print("\nClassification Report\n")

    print(
        classification_report(
            y_test,
            y_pred,
            labels=labels,
            target_names=target_names,
            zero_division=0
        )
    )

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred, labels=labels)

    print("\nConfusion Matrix Shape:", cm.shape)

    # Save results
    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Training Time (s)": round(training_time, 2)
    })

    return y_pred

In [23]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=len(label_encoder.classes_),

    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,

    subsample=0.8,
    colsample_bytree=0.8,

    tree_method="hist",
    device="cuda",

    eval_metric="mlogloss",

    random_state=42,
    n_jobs=-1
)

In [25]:
evaluate_model(
    xgb,
    "XGBoost"
)

KeyboardInterrupt: 

In [27]:
import torch
print("CUDA Available:", torch.cuda.is_available())

CUDA Available: True


In [ ]:
!nvidia-smi

In [26]:
print("Hello")

Hello


In [32]:
print(X_train.shape)

(112, 4)


In [33]:
print(X_train.dtypes.value_counts())

AttributeError: 'numpy.ndarray' object has no attribute 'dtypes'

In [34]:
print(X_train.select_dtypes(include=["object"]).columns)

AttributeError: 'numpy.ndarray' object has no attribute 'select_dtypes'

In [35]:
print(X_train.select_dtypes(include=["category"]).columns)

AttributeError: 'numpy.ndarray' object has no attribute 'select_dtypes'

In [36]:
print(type(X_train))
print(type(y_train))

print(X_train.shape)
print(y_train.shape)

print(X_train.dtype)

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
(112, 4)
(112,)
float64


In [37]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(120, 4)
(30, 4)


In [38]:
X = df.drop("Label", axis=1)
y = df["Label"]

In [40]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(1785444, 77)
(446362, 77)


In [41]:
print(type(X_train))
print(type(y_train))

<class 'pandas.DataFrame'>
<class 'pandas.Series'>


In [42]:
import numpy as np

print("NaN values:", X_train.isna().sum().sum())
print("Infinite values:", np.isinf(X_train.to_numpy()).sum())

NaN values: 0
Infinite values: 0


In [43]:
print(X_train.dtypes.value_counts())

int32      26
float32    22
int8       19
int16       7
float64     2
int64       1
Name: count, dtype: int64


In [44]:
X_small = X_train.sample(10000, random_state=42)
y_small = y_train.loc[X_small.index]

xgb.fit(X_small, y_small)

print("Training successful!")

AttributeError: module 'xgboost' has no attribute 'fit'

In [45]:
print(type(xgb))

<class 'module'>


In [46]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    objective="multi:softprob",
    num_class=len(label_encoder.classes_),

    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,

    subsample=0.8,
    colsample_bytree=0.8,

    tree_method="hist",
    device="cuda",

    eval_metric="mlogloss",
    random_state=42
)

In [47]:
X_small = X_train.sample(10000, random_state=42)
y_small = y_train.loc[X_small.index]

xgb_model.fit(X_small, y_small)

print("Training Successful!")

ValueError: Invalid classes inferred from unique values of `y`.  Expected: [ 0  1  2  3  4  5  6  7  8  9 10 11 12], got [ 0  1  2  3  4  5  6  7  9 10 11 12 14]

In [48]:
from sklearn.model_selection import train_test_split

X_small, _, y_small, _ = train_test_split(
    X_train,
    y_train,
    train_size=10000,
    stratify=y_train,
    random_state=42
)

xgb_model.fit(X_small, y_small)

ValueError: Invalid classes inferred from unique values of `y`.  Expected: [ 0  1  2  3  4  5  6  7  8  9 10 11], got [ 0  1  2  3  4  5  6  7 10 11 12 14]

In [49]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    objective="multi:softprob",
    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    device="cuda",
    eval_metric="mlogloss",
    random_state=42
)

In [50]:
from sklearn.model_selection import train_test_split

X_small, _, y_small, _ = train_test_split(
    X_train,
    y_train,
    train_size=10000,
    stratify=y_train,
    random_state=42
)

xgb_model.fit(X_small, y_small)

ValueError: Invalid classes inferred from unique values of `y`.  Expected: [ 0  1  2  3  4  5  6  7  8  9 10 11], got [ 0  1  2  3  4  5  6  7 10 11 12 14]

In [51]:
print(np.sort(y_train.unique()))

[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14]


In [52]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    objective="multi:softprob",
    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    device="cuda",
    eval_metric="mlogloss",
    random_state=42
)

In [53]:
evaluate_model(
    xgb_model,
    "XGBoost"
)

XGBoost
Training Time : 107.10 sec
Accuracy      : 0.998772
Precision     : 0.998734
Recall        : 0.998772
F1 Score      : 0.998740

Classification Report

                            precision    recall  f1-score   support

                    Benign       1.00      1.00      1.00    379064
                       Bot       0.82      0.66      0.73       288
                      DDoS       1.00      1.00      1.00     25603
             DoS GoldenEye       0.99      1.00      1.00      2057
                  DoS Hulk       1.00      1.00      1.00     34569
          DoS Slowhttptest       0.93      0.99      0.96      1046
             DoS slowloris       0.99      1.00      0.99      1077
               FTP-Patator       1.00      1.00      1.00      1186
                Heartbleed       1.00      1.00      1.00         2
              Infiltration       0.86      0.86      0.86         7
                  PortScan       0.94      0.92      0.93       391
               SSH-Patat

array([0, 0, 0, ..., 0, 0, 0], shape=(446362,))

In [54]:
import numpy as np

print("Unique labels:", np.unique(y_train))
print("Number of classes:", len(np.unique(y_train)))

Unique labels: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14]
Number of classes: 15


In [56]:
import joblib

joblib.dump(xgb_model, "xgboost_cicids2017.pkl")

['xgboost_cicids2017.pkl']

In [57]:
results_df = pd.DataFrame(results)
results_df.to_csv("results.csv", index=False)

print(results_df)

Empty DataFrame
Columns: []
Index: []


In [ ]:
LIGHTGBM

In [58]:
!pip install lightgbm -q

In [59]:
from lightgbm import LGBMClassifier

In [60]:
lgbm_model = LGBMClassifier(

    objective="multiclass",

    n_estimators=200,
    learning_rate=0.1,

    num_leaves=63,
    max_depth=-1,

    subsample=0.8,
    colsample_bytree=0.8,

    random_state=42,

    n_jobs=-1
)

In [64]:
evaluate_model(
    lgbm_model,
    "LightGBM"
)

Training LightGBM...
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.894858 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 14096
[LightGBM] [Info] Number of data points in the train set: 1785444, number of used features: 69
[LightGBM] [Info] Start training from score -0.163427
[LightGBM] [Info] Start training from score -7.348530
[LightGBM] [Info] Start training from score -2.858428
[LightGBM] [Info] Start training from score -5.379758
[LightGBM] [Info] Start training from score -2.558163
[LightGBM] [Info] Start training from score -6.056633
[LightGBM] [Info] Start training from score -6.026949
[LightGBM] [Info] Start training from score -5.930331
[LightGBM] [Info] Start training from score -12.197953
[LightGBM] [Info] Start training from score -11.027882
[LightGBM] [Info] Start training from score -7.039537
[LightGBM] 

array([0, 0, 0, ..., 0, 0, 0], shape=(446362,))

In [65]:
import joblib

joblib.dump(
    lgbm_model,
    "lightgbm_cicids2017.pkl"
)

['lightgbm_cicids2017.pkl']

In [66]:
results_df = pd.DataFrame(results)

results_df.sort_values(
    by="Accuracy",
    ascending=False
)

,Model,Accuracy,Precision,Recall,F1 Score,Training Time (s)
0,LightGBM,0.893739,0.89959,0.893739,0.879898,290.37


In [67]:
print(y_train.value_counts())

Label
0     1516250
4      138277
2      102411
3        8229
7        4745
6        4308
5        4182
11       2575
10       1565
12       1176
1        1149
14        522
9          29
13         17
8           9
Name: count, dtype: int64


In [ ]:
EXTRA TREES

In [68]:
from sklearn.ensemble import ExtraTreesClassifier

In [69]:
extra_trees_model = ExtraTreesClassifier(
    n_estimators=200,
    criterion="gini",
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    bootstrap=False,
    random_state=42,
    n_jobs=-1
)

In [70]:
evaluate_model(
    extra_trees_model,
    "Extra Trees"
)

Training Extra Trees...
Training Completed!
Making Predictions...
Prediction Completed!

Extra Trees
Training Time : 184.87 sec
Accuracy      : 0.997867
Precision     : 0.997826
Recall        : 0.997867
F1 Score      : 0.997839

Classification Report

                            precision    recall  f1-score   support

                    Benign       1.00      1.00      1.00    379064
                       Bot       0.72      0.64      0.68       288
                      DDoS       1.00      1.00      1.00     25603
             DoS GoldenEye       1.00      1.00      1.00      2057
                  DoS Hulk       0.99      0.99      0.99     34569
          DoS Slowhttptest       0.94      0.99      0.96      1046
             DoS slowloris       0.99      1.00      1.00      1077
               FTP-Patator       1.00      1.00      1.00      1186
                Heartbleed       1.00      0.50      0.67         2
              Infiltration       1.00      0.71      0.83         7

array([0, 0, 0, ..., 0, 0, 0], shape=(446362,))

In [71]:
import joblib

joblib.dump(
    extra_trees_model,
    "extra_trees_cicids2017.pkl"
)

['extra_trees_cicids2017.pkl']

In [72]:
results_df = pd.DataFrame(results)
results_df.sort_values(by="Accuracy", ascending=False)

,Model,Accuracy,Precision,Recall,F1 Score,Training Time (s)
1,Extra Trees,0.997867,0.997826,0.997867,0.997839,184.87
0,LightGBM,0.893739,0.899590,0.893739,0.879898,290.37


In [ ]:
RANDOM FOREST

In [73]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    criterion="gini",
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

In [74]:
evaluate_model(
    rf_model,
    "Random Forest"
)

Training Random Forest...
Training Completed!
Making Predictions...
Prediction Completed!

Random Forest
Training Time : 493.32 sec
Accuracy      : 0.998497
Precision     : 0.998430
Recall        : 0.998497
F1 Score      : 0.998449

Classification Report

                            precision    recall  f1-score   support

                    Benign       1.00      1.00      1.00    379064
                       Bot       0.81      0.64      0.71       288
                      DDoS       1.00      1.00      1.00     25603
             DoS GoldenEye       1.00      1.00      1.00      2057
                  DoS Hulk       1.00      1.00      1.00     34569
          DoS Slowhttptest       0.94      0.99      0.96      1046
             DoS slowloris       1.00      1.00      1.00      1077
               FTP-Patator       1.00      1.00      1.00      1186
                Heartbleed       1.00      1.00      1.00         2
              Infiltration       1.00      0.86      0.92      

array([0, 0, 0, ..., 0, 0, 0], shape=(446362,))

In [75]:
import joblib

joblib.dump(
    rf_model,
    "random_forest_cicids2017.pkl"
)

['random_forest_cicids2017.pkl']

In [76]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Accuracy",
    ascending=False
).reset_index(drop=True)

results_df

,Model,Accuracy,Precision,Recall,F1 Score,Training Time (s)
0,Random Forest,0.998497,0.998430,0.998497,0.998449,493.32
1,Extra Trees,0.997867,0.997826,0.997867,0.997839,184.87
2,LightGBM,0.893739,0.899590,0.893739,0.879898,290.37


In [77]:
results.append({
    "Model": "XGBoost",
    "Accuracy": 0.998772,
    "Precision": 0.998734,
    "Recall": 0.998772,
    "F1 Score": 0.998740,
    "Training Time (s)": 107.10
})

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="Accuracy", ascending=False).reset_index(drop=True)

results_df

,Model,Accuracy,Precision,Recall,F1 Score,Training Time (s)
0,XGBoost,0.998772,0.998734,0.998772,0.998740,107.10
1,Random Forest,0.998497,0.998430,0.998497,0.998449,493.32
2,Extra Trees,0.997867,0.997826,0.997867,0.997839,184.87
3,LightGBM,0.893739,0.899590,0.893739,0.879898,290.37


In [ ]:
HistGradientBoostingClassifier

In [78]:
from sklearn.ensemble import HistGradientBoostingClassifier

In [79]:
hgb_model = HistGradientBoostingClassifier(
    learning_rate=0.1,
    max_iter=200,
    max_depth=10,
    min_samples_leaf=20,
    random_state=42
)

In [80]:
evaluate_model(
    hgb_model,
    "HistGradientBoosting"
)

Training HistGradientBoosting...
Training Completed!
Making Predictions...
Prediction Completed!

HistGradientBoosting
Training Time : 37.39 sec
Accuracy      : 0.991556
Precision     : 0.995898
Recall        : 0.991556
F1 Score      : 0.993635

Classification Report

                            precision    recall  f1-score   support

                    Benign       1.00      0.99      1.00    379064
                       Bot       0.64      0.60      0.62       288
                      DDoS       1.00      1.00      1.00     25603
             DoS GoldenEye       0.86      0.98      0.92      2057
                  DoS Hulk       1.00      0.99      0.99     34569
          DoS Slowhttptest       0.83      0.70      0.76      1046
             DoS slowloris       0.80      0.96      0.87      1077
               FTP-Patator       0.98      0.99      0.99      1186
                Heartbleed       0.00      0.00      0.00         2
              Infiltration       0.01      0.43   

array([0, 0, 0, ..., 0, 0, 0], shape=(446362,))

In [81]:
import joblib

joblib.dump(
    hgb_model,
    "hist_gradient_boosting_cicids2017.pkl"
)

print("HistGradientBoosting model saved successfully!")

HistGradientBoosting model saved successfully!


In [82]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Accuracy",
    ascending=False
).reset_index(drop=True)

results_df

,Model,Accuracy,Precision,Recall,F1 Score,Training Time (s)
0,XGBoost,0.998772,0.998734,0.998772,0.998740,107.10
1,Random Forest,0.998497,0.998430,0.998497,0.998449,493.32
2,Extra Trees,0.997867,0.997826,0.997867,0.997839,184.87
3,HistGradientBoosting,0.991556,0.995898,0.991556,0.993635,37.39
4,LightGBM,0.893739,0.899590,0.893739,0.879898,290.37


In [83]:
results_df.to_csv(
    "model_comparison_results.csv",
    index=False
)

print("Results saved successfully!")

Results saved successfully!


In [ ]:
MLP Classifier (Neural Network)

In [84]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [85]:
from sklearn.neural_network import MLPClassifier

mlp_model = MLPClassifier(
    hidden_layer_sizes=(256, 128),
    activation="relu",
    solver="adam",
    learning_rate_init=0.001,
    batch_size=1024,
    max_iter=50,
    early_stopping=True,
    random_state=42
)

In [86]:
def evaluate_scaled_model(model, model_name):

    start = time.time()

    model.fit(X_train_scaled, y_train)

    training_time = time.time() - start

    y_pred = model.predict(X_test_scaled)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_test, y_pred, average="weighted")
    f1 = f1_score(y_test, y_pred, average="weighted")

    print("=" * 70)
    print(model_name)
    print("=" * 70)

    print(f"Training Time : {training_time:.2f} sec")
    print(f"Accuracy      : {accuracy:.6f}")
    print(f"Precision     : {precision:.6f}")
    print(f"Recall        : {recall:.6f}")
    print(f"F1 Score      : {f1:.6f}")

    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Training Time (s)": training_time
    })

    return y_pred

In [88]:
evaluate_scaled_model(
    mlp_model,
    "MLP Classifier"
)

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(


MLP Classifier
Training Time : 1981.10 sec
Accuracy      : 0.996545
Precision     : 0.996538
Recall        : 0.996545
F1 Score      : 0.996120


array([0, 0, 0, ..., 0, 0, 0], shape=(446362,))

In [89]:
import joblib

joblib.dump(
    mlp_model,
    "mlp_classifier_cicids2017.pkl"
)

print("MLP Classifier model saved successfully!")

MLP Classifier model saved successfully!


In [90]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Accuracy",
    ascending=False
).reset_index(drop=True)

results_df

,Model,Accuracy,Precision,Recall,F1 Score,Training Time (s)
0,XGBoost,0.998772,0.998734,0.998772,0.998740,107.100000
1,Random Forest,0.998497,0.998430,0.998497,0.998449,493.320000
2,Extra Trees,0.997867,0.997826,0.997867,0.997839,184.870000
3,MLP Classifier,0.996545,0.996538,0.996545,0.996120,1981.095017
4,MLP Classifier,0.995033,0.995020,0.995033,0.994608,763.279750
5,HistGradientBoosting,0.991556,0.995898,0.991556,0.993635,37.390000
6,LightGBM,0.893739,0.899590,0.893739,0.879898,290.370000


In [ ]:
TABNET

In [91]:
!pip install pytorch-tabnet -q

In [92]:
from pytorch_tabnet.tab_model import TabNetClassifier
import numpy as np

In [93]:
X_train_tabnet = X_train.values.astype(np.float32)
X_test_tabnet = X_test.values.astype(np.float32)

y_train_tabnet = y_train.values
y_test_tabnet = y_test.values

In [94]:
tabnet_model = TabNetClassifier(
    optimizer_params=dict(lr=2e-2),
    scheduler_params={"step_size":20, "gamma":0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    mask_type="entmax",
    seed=42,
    verbose=10
)

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")


In [95]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

import time

def evaluate_tabnet(model):

    start = time.time()

    model.fit(
        X_train_tabnet,
        y_train_tabnet,
        max_epochs=30,
        patience=10,
        batch_size=4096,
        virtual_batch_size=512
    )

    training_time = time.time() - start

    y_pred = model.predict(X_test_tabnet)

    accuracy = accuracy_score(y_test_tabnet, y_pred)
    precision = precision_score(
        y_test_tabnet,
        y_pred,
        average="weighted",
        zero_division=0
    )
    recall = recall_score(
        y_test_tabnet,
        y_pred,
        average="weighted"
    )
    f1 = f1_score(
        y_test_tabnet,
        y_pred,
        average="weighted"
    )

    print("="*70)
    print("TabNet")
    print("="*70)

    print(f"Training Time : {training_time:.2f} sec")
    print(f"Accuracy      : {accuracy:.6f}")
    print(f"Precision     : {precision:.6f}")
    print(f"Recall        : {recall:.6f}")
    print(f"F1 Score      : {f1:.6f}")

    print("\nClassification Report\n")

    print(classification_report(
        y_test_tabnet,
        y_pred,
        zero_division=0
    ))

    results.append({
        "Model": "TabNet",
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Training Time (s)": round(training_time,2)
    })

    return y_pred

In [96]:
evaluate_tabnet(tabnet_model)

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)


epoch 0  | loss: 0.14189 |  0:00:31s
epoch 10 | loss: 0.02551 |  0:05:34s
epoch 20 | loss: 0.02082 |  0:10:33s
TabNet
Training Time : 1213.50 sec
Accuracy      : 0.951441
Precision     : 0.960350
Recall        : 0.951441
F1 Score      : 0.952951

Classification Report

              precision    recall  f1-score   support

           0       0.99      0.95      0.97    379064
           1       0.86      0.02      0.04       288
           2       0.83      1.00      0.91     25603
           3       0.97      0.97      0.97      2057
           4       0.71      0.98      0.83     34569
           5       0.79      0.91      0.84      1046
           6       0.98      0.61      0.75      1077
           7       0.81      0.68      0.74      1186
           8       0.00      0.00      0.00         2
           9       0.00      0.00      0.00         7
          10       0.97      0.57      0.72       391
          11       1.00      0.92      0.96       644
          12       0.00    

array([0, 0, 0, ..., 0, 0, 0], shape=(446362,))

In [97]:
tabnet_model.save_model("tabnet_cicids2017")

Successfully saved model at tabnet_cicids2017.zip


'tabnet_cicids2017.zip'

In [98]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Accuracy",
    ascending=False
).reset_index(drop=True)

results_df

,Model,Accuracy,Precision,Recall,F1 Score,Training Time (s)
0,XGBoost,0.998772,0.998734,0.998772,0.998740,107.100000
1,Random Forest,0.998497,0.998430,0.998497,0.998449,493.320000
2,Extra Trees,0.997867,0.997826,0.997867,0.997839,184.870000
3,MLP Classifier,0.996545,0.996538,0.996545,0.996120,1981.095017
4,MLP Classifier,0.995033,0.995020,0.995033,0.994608,763.279750
5,HistGradientBoosting,0.991556,0.995898,0.991556,0.993635,37.390000
6,TabNet,0.951441,0.960350,0.951441,0.952951,1213.500000
7,LightGBM,0.893739,0.899590,0.893739,0.879898,290.370000


In [99]:
results_df.to_csv(
    "model_comparison_results.csv",
    index=False
)

print("Results saved successfully!")

Results saved successfully!


In [100]:
from sklearn.metrics import classification_report
import pandas as pd

report_dict = classification_report(
    y_test_tabnet,
    y_pred,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report_dict).transpose()

report_df.to_csv("TabNet_classification_report.csv")

report_df

NameError: name 'y_pred' is not defined

In [ ]:
FT- Transformer

In [1]:
!pip install pytorch-tabular -q
!pip install lightning -q

In [2]:
import pandas as pd
import numpy as np
import torch
import joblib
import time

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

from pytorch_tabular import TabularModel
from pytorch_tabular.models import FTTransformerConfig
from pytorch_tabular.config import (
    DataConfig,
    TrainerConfig,
    OptimizerConfig
)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/traitlets/config/application.py", line 1082, in launch_instance
    app.start()
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 737, in start

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/traitlets/config/application.py", line 1082, in launch_instance
    app.start()
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 737, in start

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/traitlets/config/application.py", line 1082, in launch_instance
    app.start()
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 737, in start

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: numpy.core.multiarray failed to import

In [3]:
import torch
print(torch.__version__)

2.8.0+cu128


In [4]:
import numpy as np
print(np.__version__)

2.4.6


In [5]:
import sklearn
print(sklearn.__version__)

1.9.0


In [6]:
pip install rtdl-revisiting-models

Note: you may need to restart the kernel to use updated packages.


In [3]:
import rtdl_revisiting_models as rtdl
print(rtdl.__version__)

0.0.2


In [4]:
import rtdl_revisiting_models as rtdl

print(dir(rtdl))

['Any', 'CategoricalEmbeddings', 'Dict', 'F', 'FTTransformer', 'FTTransformerBackbone', 'Iterable', 'LinearEmbeddings', 'List', 'Literal', 'MLP', 'MultiheadAttention', 'Optional', 'OrderedDict', 'Parameter', 'ResNet', 'Tensor', 'Tuple', '_CLSEmbedding', '_INTERNAL_ERROR', '_LINFORMER_KV_COMPRESSION_SHARING', '_ReGLU', '_TransformerFFNActivation', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '__version__', '_named_sequential', 'cast', 'itertools', 'math', 'nn', 'torch', 'typing', 'warnings']


In [5]:
import inspect
import rtdl_revisiting_models as rtdl

print(inspect.signature(rtdl.FTTransformer))

(*, n_cont_features: int, cat_cardinalities: List[int], _is_default: bool = False, **backbone_kwargs) -> None


In [6]:
import inspect
import rtdl_revisiting_models as rtdl

print([x for x in dir(rtdl.FTTransformer) if not x.startswith("_")])

['T_destination', 'add_module', 'apply', 'bfloat16', 'buffers', 'call_super_init', 'children', 'compile', 'cpu', 'cuda', 'double', 'dump_patches', 'eval', 'extra_repr', 'float', 'forward', 'get_buffer', 'get_default_kwargs', 'get_extra_state', 'get_parameter', 'get_submodule', 'half', 'ipu', 'load_state_dict', 'make_default_optimizer', 'make_parameter_groups', 'modules', 'mtia', 'named_buffers', 'named_children', 'named_modules', 'named_parameters', 'parameters', 'register_backward_hook', 'register_buffer', 'register_forward_hook', 'register_forward_pre_hook', 'register_full_backward_hook', 'register_full_backward_pre_hook', 'register_load_state_dict_post_hook', 'register_load_state_dict_pre_hook', 'register_module', 'register_parameter', 'register_state_dict_post_hook', 'register_state_dict_pre_hook', 'requires_grad_', 'set_extra_state', 'set_submodule', 'share_memory', 'state_dict', 'to', 'to_empty', 'train', 'type', 'xpu', 'zero_grad']


In [7]:
import rtdl_revisiting_models as rtdl

print(rtdl.FTTransformer.get_default_kwargs())

{'n_blocks': 3, 'd_block': 192, 'attention_n_heads': 8, 'attention_dropout': 0.2, 'ffn_d_hidden': None, 'ffn_d_hidden_multiplier': 1.3333333333333333, 'ffn_dropout': 0.1, 'residual_dropout': 0.0, '_is_default': True}


In [8]:
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

import rtdl_revisiting_models as rtdl

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [10]:
class CICDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.tensor(
            X.values,
            dtype=torch.float32
        )

        self.y = torch.tensor(
            y.values,
            dtype=torch.long
        )

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [11]:
train_dataset = CICDataset(X_train, y_train)
test_dataset = CICDataset(X_test, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=4096,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4096,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

NameError: name 'X_train' is not defined

In [12]:
print(globals().keys())

dict_keys(['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh', 'In', 'Out', 'get_ipython', 'exit', 'quit', 'open', '_', '__', '___', '__session__', '_i', '_ii', '_iii', '_i1', '_exit_code', '_i2', 'pd', 'np', 'torch', 'joblib', 'time', 'accuracy_score', 'precision_score', 'recall_score', 'f1_score', 'classification_report', '_i3', 'rtdl', '_i4', '_i5', 'inspect', '_i6', '_i7', '_i8', 'nn', 'Dataset', 'DataLoader', '_i9', 'device', '_i10', 'CICDataset', '_i11', '_i12'])


In [13]:
from sklearn.model_selection import train_test_split

X = df.drop("Label", axis=1)
y = df["Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

NameError: name 'df' is not defined

In [14]:
import os

for file in os.listdir():
    print(file)

y_train.pkl
.sudo_as_admin_successful
.idea
.oh-my-zsh
.local
.lightningignore
.zcompdump
.profile
y_test.pkl
.windsurf-server
X_test.pkl
tabnet_cicids2017.zip
main.py
.nvm
mlp_classifier_cicids2017.pkl
.ipython
.vimrc
.config
.gitconfig
.ssh
xgboost_cicids2017.pkl
Untitled.ipynb
.condarc
.vscode-server
.bashrc
results.csv
extra_trees_cicids2017.pkl
.cursor-server
archive.zip
random_forest_cicids2017.pkl
.docker
.lightning_studio
.hushlogin
.nv
CICIDS2017
.vscode
X_train.pkl
.jupyter
.cache
lightgbm_cicids2017.pkl
model_comparison_results.csv
.zshrc
hist_gradient_boosting_cicids2017.pkl


In [15]:
import os

for root, dirs, files in os.walk("CICIDS2017"):
    for f in files:
        print(os.path.join(root, f))

CICIDS2017/DoS-Wednesday-no-metadata.parquet
CICIDS2017/Portscan-Friday-no-metadata.parquet
CICIDS2017/Botnet-Friday-no-metadata.parquet
CICIDS2017/Bruteforce-Tuesday-no-metadata.parquet
CICIDS2017/Benign-Monday-no-metadata.parquet
CICIDS2017/DDoS-Friday-no-metadata.parquet
CICIDS2017/Infiltration-Thursday-no-metadata.parquet
CICIDS2017/WebAttacks-Thursday-no-metadata.parquet


In [16]:
import pandas as pd
import numpy as np
import os

folder = "CICIDS2017"

files = [
    os.path.join(folder, f)
    for f in os.listdir(folder)
    if f.endswith(".parquet")
]

df = pd.concat(
    [pd.read_parquet(f) for f in files],
    ignore_index=True
)

print(df.shape)
df.head()

(2313810, 78)


,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,6,38308,1,1,6,6,6,6,6.000000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
1,6,479,11,5,172,326,79,0,15.636364,31.449238,...,32,0.0,0.0,0,0,0.0,0.0,0,0,Benign
2,6,1095,10,6,3150,3150,1575,0,315.000000,632.561646,...,32,0.0,0.0,0,0,0.0,0.0,0,0,Benign
3,6,15206,17,12,3452,6660,1313,0,203.058823,425.778473,...,32,0.0,0.0,0,0,0.0,0.0,0,0,Benign
4,6,1092,9,6,3150,3152,1575,0,350.000000,694.509705,...,32,0.0,0.0,0,0,0.0,0.0,0,0,Benign


In [17]:
df.drop_duplicates(inplace=True)

print(df.shape)

(2231806, 78)


In [18]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)

In [19]:
numeric_cols = df.select_dtypes(include=np.number).columns

df[numeric_cols] = df[numeric_cols].fillna(
    df[numeric_cols].median()
)

In [20]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

df["Label"] = encoder.fit_transform(df["Label"])

In [21]:
X = df.drop("Label", axis=1)
y = df["Label"]

print(X.shape)
print(y.shape)

(2231806, 77)
(2231806,)


In [22]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [23]:
print(X_train.shape)
print(X_test.shape)

print(y_train.shape)
print(y_test.shape)

(1785444, 77)
(446362, 77)
(1785444,)
(446362,)


In [24]:
import joblib

joblib.dump(X_train, "X_train.pkl")
joblib.dump(X_test, "X_test.pkl")
joblib.dump(y_train, "y_train.pkl")
joblib.dump(y_test, "y_test.pkl")

['y_test.pkl']

In [25]:
train_dataset = CICDataset(X_train, y_train)
test_dataset = CICDataset(X_test, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=4096,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4096,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [26]:
model = rtdl.FTTransformer(

    n_cont_features=77,

    cat_cardinalities=[],

    d_out=15,

    n_blocks=3,
    d_block=192,
    attention_n_heads=8,
    attention_dropout=0.2,
    ffn_d_hidden=None,
    ffn_d_hidden_multiplier=4/3,
    ffn_dropout=0.1,
    residual_dropout=0.0,

).to(device)

In [27]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

In [28]:
epochs = 20

start = time.time()

for epoch in range(epochs):

    model.train()

    running_loss = 0.0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch,None)

        loss = criterion(outputs, y_batch)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Loss: {running_loss / len(train_loader):.4f}"
    )

training_time = time.time() - start

print(f"\nTraining Time: {training_time:.2f} seconds")

Epoch [1/20] Loss: 0.3102
Epoch [2/20] Loss: 0.1796
Epoch [3/20] Loss: 0.1634
Epoch [4/20] Loss: 0.1632
Epoch [5/20] Loss: 0.1519
Epoch [6/20] Loss: 0.1616
Epoch [7/20] Loss: 0.1560
Epoch [8/20] Loss: 0.1530
Epoch [9/20] Loss: 0.1315
Epoch [10/20] Loss: 0.1243
Epoch [11/20] Loss: 0.1282
Epoch [12/20] Loss: 0.1241
Epoch [13/20] Loss: 0.1225
Epoch [14/20] Loss: 0.1191
Epoch [15/20] Loss: 0.1342
Epoch [16/20] Loss: 0.1307
Epoch [17/20] Loss: 0.1265
Epoch [18/20] Loss: 0.1191
Epoch [19/20] Loss: 0.1117
Epoch [20/20] Loss: 0.1091

Training Time: 10716.78 seconds


In [29]:
model.eval()

y_true = []
y_pred = []

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        X_batch = X_batch.to(device)

        outputs = model(X_batch, None)

        predictions = torch.argmax(outputs, dim=1)

        y_pred.extend(predictions.cpu().numpy())
        y_true.extend(y_batch.numpy())

In [30]:
accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

print(f"Accuracy : {accuracy:.6f}")
print(f"Precision: {precision:.6f}")
print(f"Recall   : {recall:.6f}")
print(f"F1 Score : {f1:.6f}")
print(f"Training Time: {training_time:.2f} seconds")

Accuracy : 0.964034
Precision: 0.961708
Recall   : 0.964034
F1 Score : 0.961841
Training Time: 10716.78 seconds


In [31]:
print(classification_report(
    y_true,
    y_pred,
    zero_division=0
))

              precision    recall  f1-score   support

           0       0.98      0.99      0.98    379064
           1       0.99      0.40      0.57       288
           2       0.91      0.89      0.90     25603
           3       0.90      0.50      0.65      2057
           4       0.89      0.83      0.86     34569
           5       0.78      0.59      0.67      1046
           6       0.85      0.32      0.46      1077
           7       0.61      0.37      0.46      1186
           8       0.00      0.00      0.00         2
           9       0.00      0.00      0.00         7
          10       0.81      0.59      0.68       391
          11       0.96      0.41      0.57       644
          12       0.00      0.00      0.00       294
          13       0.00      0.00      0.00         4
          14       0.00      0.00      0.00       130

    accuracy                           0.96    446362
   macro avg       0.58      0.39      0.45    446362
weighted avg       0.96   

In [32]:
report = classification_report(
    y_true,
    y_pred,
    output_dict=True,
    zero_division=0
)

pd.DataFrame(report).transpose().to_csv(
    "FTTransformer_report.csv"
)

In [33]:
predictions = pd.DataFrame({
    "Actual": y_true,
    "Predicted": y_pred
})

predictions.to_csv(
    "FTTransformer_predictions.csv",
    index=False
)

In [34]:
torch.save(
    model.state_dict(),
    "FTTransformer_cicids2017.pth"
)

In [35]:
results.append({
    "Model": "FTTransformer",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1 Score": f1,
    "Training Time (s)": training_time
})

NameError: name 'results' is not defined

In [ ]:
pd.DataFrame(results).to_csv(
    "model_comparison_results.csv",
    index=False
)

In [39]:
import inspect

print(inspect.signature(model.forward))

(x_cont: Optional[torch.Tensor], x_cat: Optional[torch.Tensor]) -> torch.Tensor


In [40]:
X_batch, y_batch = next(iter(train_loader))

X_batch = X_batch.to(device)

outputs = model(X_batch, None)

print(outputs.shape)

OutOfMemoryError: CUDA out of memory. Tried to allocate 234.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 204.81 MiB is free. Including non-PyTorch memory, this process has 14.36 GiB memory in use. Of the allocated memory 13.91 GiB is allocated by PyTorch, and 332.81 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [36]:
results.append({
    "Model": "FT-Transformer",
    "Accuracy": 0.964034,
    "Precision": 0.961708,
    "Recall": 0.964034,
    "F1 Score": 0.961841,
    "Training Time (s)": 10716.78
})

NameError: name 'results' is not defined

In [37]:
import pandas as pd

comparison = pd.read_csv("model_comparison_results.csv")

new_row = pd.DataFrame([{
    "Model": "FT-Transformer",
    "Accuracy": 0.964034,
    "Precision": 0.961708,
    "Recall": 0.964034,
    "F1 Score": 0.961841,
    "Training Time (s)": 10716.78
}])

comparison = pd.concat([comparison, new_row], ignore_index=True)

comparison.to_csv("model_comparison_results.csv", index=False)

print(comparison)

                  Model  Accuracy  Precision    Recall  F1 Score  \
0               XGBoost  0.998772   0.998734  0.998772  0.998740   
1         Random Forest  0.998497   0.998430  0.998497  0.998449   
2           Extra Trees  0.997867   0.997826  0.997867  0.997839   
3        MLP Classifier  0.996545   0.996538  0.996545  0.996120   
4        MLP Classifier  0.995033   0.995020  0.995033  0.994608   
5  HistGradientBoosting  0.991556   0.995898  0.991556  0.993635   
6                TabNet  0.951441   0.960350  0.951441  0.952951   
7              LightGBM  0.893739   0.899590  0.893739  0.879898   
8        FT-Transformer  0.964034   0.961708  0.964034  0.961841   

   Training Time (s)  
0         107.100000  
1         493.320000  
2         184.870000  
3        1981.095017  
4         763.279750  
5          37.390000  
6        1213.500000  
7         290.370000  
8       10716.780000  


In [38]:
torch.save(
    model.state_dict(),
    "ft_transformer_cicids2017.pth"
)

print("Model saved successfully!")

Model saved successfully!
